In [5]:
with open("D:/RecSeach/PENS/news.tsv",encoding="utf-8") as f:
    for i,line in enumerate(f,1):
        if i<=12:
            print(i,repr(line[:300]))

1 'News ID\tCategory\tTopic\tHeadline\tNews body\tTitle entity\tEntity content\n'
2 "N10000\tsports\tsoccer\tPredicting Atlanta United's lineup against Columbus Crew in the U.S. Open Cup\tOnly FIVE internationals allowed, count em, FIVE! So first off we should say, per our usual Atlanta United lineup predictions, this will be wrong. Why will it be wrong? Well, aside from the obvious, we"
3 'N10001\tnews\tnewspolitics\tMitch McConnell: DC statehood push is \'full bore socialism\'\t"WASHINGTON -- Senate Majority Leader Mitch McConnell criticized the push to grant statehood to Puerto Rico and Washington, D.C. in an interview on FOX News with Laura Ingraham. ""They plan to make the District of C'
4 'N10002\tnews\tnewsus\tHome In North Highlands Damaged By Fire\tNORTH HIGHLANDS (CBS13)   Fire damaged a home in North Highlands overnight. (credit: Sacramento Metro Fire Dept.) The fire burned at a mobile home on Eureka Lane. Firefighters from Sacramento Metro Fire Department quickly arrived on

In [ ]:
import pandas as pd

df = pd.read_csv("D:/RecSeach/PENS/news.tsv",sep='\t',encoding="utf-8")


<class 'pandas.core.indexes.base.Index'>


In [33]:
df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
# ['news_id', 'category', 'topic', 'headline', 'news_body', 'title_entity', 'entity_content']
print(df.columns)

Index(['news_id', 'category', 'topic', 'headline', 'news_body', 'title_entity',
       'entity_content'],
      dtype='object')


In [36]:
print(df.dtypes)
print(df.isna().sum())

news_id           object
category          object
topic             object
headline          object
news_body         object
title_entity      object
entity_content    object
dtype: object
news_id           0
category          0
topic             0
headline          0
news_body         0
title_entity      0
entity_content    0
dtype: int64


In [35]:
for col in ["headline", "news_body", "category", "topic"]:
    df[col] = df[col].fillna("").astype(str)

In [37]:
df = df[df["news_id"].notna() & (df["news_id"] != "")]

In [49]:
print(df.loc[4,'headline'])

Today in History: Aug 1


In [23]:
with open('D:/RecSeach/Search/entity.json','w',encoding='utf-8') as f:
    f.write(df.loc[0,'Entity content'])

In [51]:
print('新闻数量:',df.shape[0])


新闻数量: 111527


In [32]:
import ast
ec = ast.literal_eval(df.loc[8, "Entity content"])
print(ec.keys())

dict_keys(['Mexico', 'Restaurant'])


In [40]:
import re
import html

def clean_text(s):
    if not isinstance(s, str):
        return ""
    # HTML 实体：&amp; → &
    s = html.unescape(s)
    # HTML 标签
    s = re.sub(r"<[^>]+>", " ", s)
    # URL
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    # 邮箱
    s = re.sub(r"\S+@\S+", " ", s)
    # 多余转义
    s = s.replace("\\n", " ").replace("\\t", " ").replace("\\r", " ")
    # 连续空白 → 单空格
    s = re.sub(r"\s+", " ", s)
    return s.strip()

df["headline"]  = df["headline"].apply(clean_text)
df["news_body"] = df["news_body"].apply(clean_text)

In [41]:
# 完全重复的 news_id（不该有，有就是数据错误）
dup_id = df["news_id"].duplicated().sum()
print("重复 news_id:", dup_id)
df = df.drop_duplicates(subset="news_id", keep="first")

# 正文完全相同的（可能是转载）
dup_body = df.duplicated(subset=["headline", "news_body"]).sum()
print("重复内容:", dup_body)
df = df.drop_duplicates(subset=["headline", "news_body"], keep="first")

df = df.reset_index(drop=True)

重复 news_id: 0
重复内容: 3


In [42]:
import re

def tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    # 只保留字母数字和撇号
    tokens = re.findall(r"[a-z0-9']+", text)
    # 去首尾撇号
    tokens = [t.strip("'") for t in tokens]
    # 去空
    tokens = [t for t in tokens if t]
    return tokens

df["headline_tokens"] = df["headline"].apply(tokenize)
df["body_tokens"]     = df["news_body"].apply(tokenize)

In [50]:
# ['news_id', 'category', 'topic', 'headline', 'news_body', 'title_entity', 'entity_content']
newdf=df[['news_id',"headline_tokens","body_tokens"]]
newdf.to_parquet('tokens.parquet', index=False)